# VeloceReduction — one observing night

The notebook is deliberately thin. Each cell exposes one major stage for QA; the same sequence is available non-interactively through `pipeline.reduce_night()`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from velocereduction import __version__, ReductionConfig
from velocereduction import observations, detector, tramlines, flat, extraction, wavelength, science
from velocereduction.config import prepare_reduction, setup_logging

night = "001122"
# night = "260703"
config = ReductionConfig(
    night=night,
    extraction_mode="summed",   # "summed" for summing information along cross-dispersion direction; "fibre" for extracting each science fibre separately
    diagnostics="full",
    log_level="DEBUG",
    overwrite=False,
    use_poisson_variance=True,
)
paths = prepare_reduction(config, __version__)
logger = setup_logging(config, paths)
print(paths.root)


## 1. Observations and detector registration


In [ ]:
reduction_input = observations.identify_observations(config, paths)
detector_shifts = detector.measure_detector_shifts(reduction_input, config, paths)
display(reduction_input)
display(detector_shifts)


## 2. Master Flat and nightly tramlines


In [ ]:
master_flat = flat.create_master_flat(reduction_input, config, paths)
nightly_tramlines = tramlines.fit_nightly_tramlines(
    reduction_input, master_flat, detector_shifts, config, paths
)
display(nightly_tramlines[:5])


## 3. Flat products and fibre geometry


In [ ]:
flat_products = flat.create_flat_products(master_flat, nightly_tramlines, config, paths)

if config.extraction_mode == "fibre":
    name = next(iter(flat_products))
    g = flat_products[name].geometry

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(g.sampled_x, g.sampled_sigma, ".", label="sampled")
    ax.plot(g.sigma, label="smooth")
    ax.set(xlabel="dispersion pixel", ylabel=r"$\sigma$ / pixel", title=name)
    ax.legend(); plt.show()

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(g.sampled_x, g.sampled_separation, ".", label="sampled")
    ax.plot(g.separation, label="smooth")
    ax.set(xlabel="dispersion pixel", ylabel="fibre separation / pixel", title=name)
    ax.legend(); plt.show()


## 4. Extract calibrations and fit wavelength model


In [ ]:
model_file = paths.wavelength_mode / "wavelength_model.fits"
if model_file.exists() and not config.overwrite:
    calibration_exposures = None
    wavelength_model = wavelength.load_model(model_file)
else:
    calibration_exposures = extraction.extract_calibration_exposures(
        reduction_input, nightly_tramlines, flat_products, config
    )
    extraction.save_calibration_exposures(
        calibration_exposures, paths.wavelength_mode / "extracted", overwrite=True
    )
    wavelength_model = wavelength.build_night_model(
        calibration_exposures, detector_shifts, config, paths
    )

display(wavelength.model_summary(wavelength_model))


The wavelength hierarchy is:

- summed FibTh: absolute science-bundle anchor;
- SimLC on CCD2/3: independent high-precision temporal drift;
- per-fibre FibTh: low-order residual correction for each of the 19 science fibres.


## 5. Science extraction


In [ ]:
science_exposures = science.extract_science_exposures(
    reduction_input, nightly_tramlines, flat_products,
    wavelength_model, config, paths
)
print(f"{len(science_exposures)} science CCD exposures")

if science_exposures:
    exposure = science_exposures[0]
    order = exposure.orders[len(exposure.orders) // 2]
    plt.figure(figsize=(10, 3))
    plt.plot(order.barycentric_wavelength_nm, order.flux)
    plt.xlabel("Barycentric wavelength / nm")
    plt.ylabel("Flux")
    plt.title(f"{exposure.object_name} — CCD{exposure.ccd}, order {order.order}")
    plt.show()


## Automatic QA products

With `diagnostics='basic'`, each stage above has already written the compact nightly QA figures to `figures/`. `full` additionally writes per-order diagnostics to `debug/`.


In [ ]:
if config.diagnostics != 'none':
    for filename in sorted(paths.figures.rglob('*.png')):
        print(filename.relative_to(paths.root))


## One-call equivalent


In [ ]:
# from velocereduction import pipeline
# state = pipeline.reduce_night(config, version=__version__)
